# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [6]:
print("QUESTION: Can a model rank which pages will lose search clicks better than simply ranking by earlier clicks?")
print("DECISION: which pages a content or SEO editor reviews first. A wrong call costs review time, or a bad edit to a page that already earns clicks.")
print("LABEL: declined = clicks in the last 30 days are below clicks in the previous 30 days.")
print("GRAIN: one row = one content page.")
print("LANE: Refresh / Content Opportunity Scoring. Output: a ranked review queue with reason codes.")

QUESTION: Can a model rank which pages will lose search clicks better than simply ranking by earlier clicks?
DECISION: which pages a content or SEO editor reviews first. A wrong call costs review time, or a bad edit to a page that already earns clicks.
LABEL: declined = clicks in the last 30 days are below clicks in the previous 30 days.
GRAIN: one row = one content page.
LANE: Refresh / Content Opportunity Scoring. Output: a ranked review queue with reason codes.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [7]:
import os, json, subprocess
import numpy as np
import pandas as pd

# Get the repo (the starter table lives in data/raw/). Skip the clone if it is already there.
if not os.path.exists("flyrank-ml-internship"):
    subprocess.run(["git", "clone", "-q", "https://github.com/rekhasreebandaru/flyrank-ml-internship.git"], check=True)

def p_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

df = pd.read_csv("flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")
print("Table used: the starter table content_refresh_anonymized.csv (the only table). Rows:", len(df))

# Only pages that COULD decline (clicks in the earlier 30 days). Features come from the earlier window only.
elig = df[df["clicks_prev_30d"] > 0].copy()
elig["declined"] = (elig["clicks_last_30d"] < elig["clicks_prev_30d"]).astype(int)
elig["early_clicks"] = elig["clicks_90d"] - elig["clicks_last_30d"] - elig["clicks_prev_30d"]
elig["early_impr"] = elig["impressions_90d"] - elig["impressions_last_30d"] - elig["impressions_prev_30d"]
elig["prev_ctr"] = np.where(elig["impressions_prev_30d"] > 0,
                            elig["clicks_prev_30d"] / elig["impressions_prev_30d"], 0)
feat2 = ["clicks_prev_30d", "impressions_prev_30d", "sessions_prev_30d", "prev_ctr",
         "search_volume", "competition", "cpc", "word_count", "content_age_days",
         "early_clicks", "early_impr"]
elig = elig.dropna(subset=feat2 + ["client_id"])
print("Pages that could decline (clicks in the earlier 30 days):", len(elig), "from", elig["client_id"].nunique(), "clients")
print()
print("FEATURES (earlier window only):", feat2)
print()
print("EXCLUDED ON PURPOSE:")
print("- ctr, engagement_rate, scroll_rate, avg_position, position_tier: trailing 90-day measures that contain the label window")
print("- days_since_last_update: an update in the last 30 days falls inside the label window")
print("- trend_pct, trend_direction: label-related by construction")
print("- client_id, content_id: pseudonyms, used for grouping only, never as features")
print("- pages with no clicks in the earlier 30 days: they cannot decline")

Table used: the starter table content_refresh_anonymized.csv (the only table). Rows: 30000
Pages that could decline (clicks in the earlier 30 days): 8462 from 26 clients

FEATURES (earlier window only): ['clicks_prev_30d', 'impressions_prev_30d', 'sessions_prev_30d', 'prev_ctr', 'search_volume', 'competition', 'cpc', 'word_count', 'content_age_days', 'early_clicks', 'early_impr']

EXCLUDED ON PURPOSE:
- ctr, engagement_rate, scroll_rate, avg_position, position_tier: trailing 90-day measures that contain the label window
- days_since_last_update: an update in the last 30 days falls inside the label window
- trend_pct, trend_direction: label-related by construction
- client_id, content_id: pseudonyms, used for grouping only, never as features
- pages with no clicks in the earlier 30 days: they cannot decline


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [8]:
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Grouped split by client: the model is always tested on clients it has not seen
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
i_tr, i_te = next(gss2.split(elig, groups=elig["client_id"]))
tr2, te2 = elig.iloc[i_tr].copy(), elig.iloc[i_te].copy()
rf2 = RandomForestClassifier(random_state=42, n_estimators=200).fit(tr2[feat2], tr2["declined"])
te2["score"] = rf2.predict_proba(te2[feat2])[:, 1]

# Cutoffs fixed from TRAINING data only
oof = cross_val_predict(RandomForestClassifier(random_state=42, n_estimators=200),
                        tr2[feat2], tr2["declined"], groups=tr2["client_id"],
                        cv=GroupKFold(n_splits=5), method="predict_proba")[:, 1]
cut2 = np.quantile(oof, 0.80)
traffic_cut = tr2["clicks_prev_30d"].median()

te2["high_risk"] = te2["score"] >= cut2
te2["meaningful_traffic"] = te2["clicks_prev_30d"] > traffic_cut
te2["clicks_lost"] = (te2["clicks_prev_30d"] - te2["clicks_last_30d"]).clip(lower=0)
te2["reason_code"] = np.select(
    [te2["high_risk"] & te2["meaningful_traffic"],
     te2["high_risk"] & ~te2["meaningful_traffic"],
     ~te2["high_risk"] & te2["meaningful_traffic"]],
    ["meaningful_traffic_high_risk", "small_traffic_high_risk", "meaningful_traffic_low_risk"],
    default="small_traffic_low_risk")
te2["action"] = te2["reason_code"].map({
    "meaningful_traffic_high_risk": "review_first", "small_traffic_high_risk": "monitor",
    "meaningful_traffic_low_risk": "routine_check", "small_traffic_low_risk": "no_action"})
rank_order = {"review_first": 0, "monitor": 1, "routine_check": 2, "no_action": 3}
te2["_o"] = te2["action"].map(rank_order)
ranked_queue = te2.sort_values(["_o", "score"], ascending=[True, False]).drop(columns="_o")

print("ASSUMPTIONS AND DESIGN")
print("- Label: declined = clicks_last_30d < clicks_prev_30d, on pages with earlier clicks")
print("- Model: Random Forest, 200 trees, seed 42. Baselines: the base rate, and ranking by earlier clicks")
print("- Split: grouped by client_id, 75% train and 25% test. Train clients:", tr2['client_id'].nunique(), "| test clients:", te2['client_id'].nunique())
print(f"- High risk: score >= {cut2:.3f} (80th percentile of out-of-fold TRAINING scores). Meaningful traffic: more than {traffic_cut:.0f} earlier clicks (training median)")
print("- Leakage checks: every feature predates the label window; no product flags or label-derived columns; grouped split;")
print("  train with and without the top feature. That last test passed in the first version and still missed the overlapping windows.")

ASSUMPTIONS AND DESIGN
- Label: declined = clicks_last_30d < clicks_prev_30d, on pages with earlier clicks
- Model: Random Forest, 200 trees, seed 42. Baselines: the base rate, and ranking by earlier clicks
- Split: grouped by client_id, 75% train and 25% test. Train clients: 19 | test clients: 7
- High risk: score >= 0.705 (80th percentile of out-of-fold TRAINING scores). Meaningful traffic: more than 4 earlier clicks (training median)
- Leakage checks: every feature predates the label window; no product flags or label-derived columns; grouped split;
  train with and without the top feature. That last test passed in the first version and still missed the overlapping windows.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [9]:
print("FIRST VERSION (overlapping 90-day features, all pages scored). Numbers from the ML-09 notebook:")
print(pd.DataFrame({"split": ["Random", "Grouped by client"], "precision@20": [0.85, 0.30], "precision@50": [0.88, 0.34]}).to_string(index=False))
print("A random split can overstate skill by a wide margin.")
print()

m = te2[te2["meaningful_traffic"]]
base_all, base_m = te2["declined"].mean(), m["declined"].mean()
print(f"REBUILT MODEL. Test: {len(te2)} pages, {te2['client_id'].nunique()} clients, base decline rate {base_all:.3f}")
print(f"Pages with meaningful traffic: n={len(m)}, base rate {base_m:.3f}")
rows = []
for k in [20, 50, 100]:
    rows.append({"K": k, "model": p_at_k(m["score"].values, m["declined"].values, k),
                 "rank by earlier clicks": p_at_k(m["clicks_prev_30d"].values, m["declined"].values, k)})
print(pd.DataFrame(rows).round(3).to_string(index=False))
auc_m, auc_r = roc_auc_score(m["declined"], m["score"]), roc_auc_score(m["declined"], m["clicks_prev_30d"])

rng = np.random.default_rng(42)
diffs = []
for _ in range(300):
    mm = m.iloc[rng.integers(0, len(m), len(m))]
    if mm["declined"].nunique() < 2:
        continue
    diffs.append(roc_auc_score(mm["declined"], mm["score"]) - roc_auc_score(mm["declined"], mm["clicks_prev_30d"]))
lo, hi = np.percentile(diffs, [5, 95])
print(f"AUC: model {auc_m:.3f}, earlier clicks {auc_r:.3f}. Model minus earlier clicks, 90% range {lo:.3f} to {hi:.3f}")
print("(That range resamples pages, not clients, so it understates the real uncertainty.)")
fl, un = m[m["high_risk"]], m[~m["high_risk"]]
print(f"Flagged decline rate {fl['declined'].mean():.3f} (n={len(fl)}) vs unflagged {un['declined'].mean():.3f} (n={len(un)})")
print()
q = pd.qcut(te2["clicks_prev_30d"].rank(method="first"), 4, labels=["lowest", "second", "third", "highest"])
print("Decline rate by earlier-click quarter (all held-out pages):")
print(te2.groupby(q, observed=True)["declined"].mean().round(3).to_string())
print()
print("READING: where pages have meaningful traffic, the model did not rank declines better than a simple rule in this split.")
print("Pages with more earlier clicks declined more often, which fits regression to the mean (not tested here).")

FIRST VERSION (overlapping 90-day features, all pages scored). Numbers from the ML-09 notebook:
            split  precision@20  precision@50
           Random          0.85          0.88
Grouped by client          0.30          0.34
A random split can overstate skill by a wide margin.

REBUILT MODEL. Test: 3568 pages, 7 clients, base decline rate 0.547
Pages with meaningful traffic: n=1633, base rate 0.627
  K  model  rank by earlier clicks
 20   0.85                    0.90
 50   0.72                    0.84
100   0.71                    0.79
AUC: model 0.542, earlier clicks 0.608. Model minus earlier clicks, 90% range -0.098 to -0.038
(That range resamples pages, not clients, so it understates the real uncertainty.)
Flagged decline rate 0.659 (n=208) vs unflagged 0.622 (n=1425)

Decline rate by earlier-click quarter (all held-out pages):
clicks_prev_30d
lowest     0.443
second     0.511
third      0.530
highest    0.704

READING: where pages have meaningful traffic, the model did no

## 5. Limitations

*What this work cannot claim.*

In [10]:
print("LIMITATIONS")
print(f"- One snapshot, one split, {te2['client_id'].nunique()} held-out clients. A different split could move these numbers.")
print("- The label is a 30-day click drop. Noise and seasonality can produce it.")
print("- Only pages with earlier clicks are scored. Pages with none are out of scope.")
print("- The rise in decline rate with earlier clicks is consistent with regression to the mean, which was not tested.")
print("- Nothing here shows that refreshing a page reverses a decline. That needs a controlled experiment.")
print("- The decay finding is descriptive. A recent update falls inside the label window.")
print("- A recommendation is wrong if the drop is tracking noise, seasonality or a one-off spike, or if the page is meant to rank low.")
print("- The overlapping-window and base-rate problems in the first version were found and corrected after the first results.")

LIMITATIONS
- One snapshot, one split, 7 held-out clients. A different split could move these numbers.
- The label is a 30-day click drop. Noise and seasonality can produce it.
- Only pages with earlier clicks are scored. Pages with none are out of scope.
- The rise in decline rate with earlier clicks is consistent with regression to the mean, which was not tested.
- Nothing here shows that refreshing a page reverses a decline. That needs a controlled experiment.
- The decay finding is descriptive. A recent update falls inside the label window.
- A recommendation is wrong if the drop is tracking noise, seasonality or a one-off spike, or if the page is meant to rank low.
- The overlapping-window and base-rate problems in the first version were found and corrected after the first results.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [11]:
summ = (ranked_queue.groupby(["action", "reason_code"])
        .agg(pages=("declined", "size"), decline_rate=("declined", "mean"),
             avg_clicks_lost=("clicks_lost", "mean")).reset_index())
summ["_o"] = summ["action"].map(rank_order)
print("RANKED ACTIONS AND REASON CODES (review_first at the top of the queue)")
print(summ.sort_values("_o").drop(columns="_o").round(3).to_string(index=False))
print()
print("TOP 10 OF THE QUEUE")
print(ranked_queue[["content_id", "clicks_prev_30d", "score", "reason_code", "action"]].head(10).to_string(index=False))
print()
print("RANKED RECOMMENDATIONS")
print("1. Review pages with the most earlier clicks first, after confirming the drop is real. Ranking by earlier clicks matched or beat the model among pages with meaningful traffic.")
print("2. Use the model flag as a secondary note, not a ranker, for high-traffic pages. The gap between flagged and unflagged pages was small.")
print("3. Treat small-traffic flags as monitor-only. Those declines cost about one click.")
print("4. Do not automate edits, deletions or client promises. No test here shows that a refresh recovers traffic.")
print("5. Run a controlled refresh test before claiming any effect.")
print("6. Re-test any new model against ranking by earlier clicks on a fresh month and on new clients.")

RANKED ACTIONS AND REASON CODES (review_first at the top of the queue)
       action                  reason_code  pages  decline_rate  avg_clicks_lost
 review_first meaningful_traffic_high_risk    208         0.659           16.553
      monitor      small_traffic_high_risk    410         0.741            1.032
routine_check  meaningful_traffic_low_risk   1425         0.622            8.045
    no_action       small_traffic_low_risk   1525         0.409            0.606

TOP 10 OF THE QUEUE
          content_id  clicks_prev_30d  score                  reason_code       action
content_47b8b12d581e              164  0.915 meaningful_traffic_high_risk review_first
content_b3f2ac089cc6                6  0.905 meaningful_traffic_high_risk review_first
content_299ec9fd826c                5  0.900 meaningful_traffic_high_risk review_first
content_649c63664f77               40  0.895 meaningful_traffic_high_risk review_first
content_e44c7b688272                7  0.890 meaningful_traffic_high

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [12]:
import matplotlib
import matplotlib.pyplot as plt

os.makedirs("docs/img", exist_ok=True)
os.makedirs("work/outputs", exist_ok=True)
BLUE, GREY, INK = "#1F5A7A", "#9AA3B2", "#1B2330"
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

def label_bars(ax, bars, fmt="{:.2f}", inside=False):
    for b in bars:
        if inside:
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() - 0.025, fmt.format(b.get_height()),
                    ha="center", va="top", fontsize=11, color="white", fontweight="bold")
        else:
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.012, fmt.format(b.get_height()),
                    ha="center", va="bottom", fontsize=10, color=INK)

def legend_on_top(ax, n):
    ax.legend(frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=n)

w = 0.36
# Chart 1: first version under two splits (numbers from the ML-09 notebook)
fig, ax = plt.subplots(figsize=(7.2, 3.8))
x = np.arange(2)
b1 = ax.bar(x - w/2, [0.85, 0.88], w, color=GREY, label="Random split")
b2 = ax.bar(x + w/2, [0.30, 0.34], w, color=BLUE, label="Grouped by client")
label_bars(ax, b1); label_bars(ax, b2)
ax.set_xticks(x); ax.set_xticklabels(["precision@20", "precision@50"])
ax.set_ylabel("precision (higher is better)"); ax.set_ylim(0, 1.05)
legend_on_top(ax, 2)
plt.tight_layout(); plt.savefig("docs/img/chart1_split_gap.png", dpi=160); plt.close()

# Chart 2: decline rate by earlier-click quarter
q = pd.qcut(te2["clicks_prev_30d"].rank(method="first"), 4,
            labels=["Lowest quarter", "Second", "Third", "Highest quarter"])
g = te2.groupby(q, observed=True).agg(rate=("declined", "mean"), med=("clicks_prev_30d", "median"))
fig, ax = plt.subplots(figsize=(7.2, 3.8))
bars = ax.bar(range(4), g["rate"], color=BLUE)
label_bars(ax, bars, inside=True)
ax.axhline(te2["declined"].mean(), color=INK, linestyle="--", linewidth=1, label="decline rate across all pages")
legend_on_top(ax, 1)
ax.set_xticks(range(4))
ax.set_xticklabels([f"{n}\n(median {int(v)} click{'s' if v != 1 else ''})" for n, v in zip(g.index, g["med"])], fontsize=9)
ax.set_ylabel("share of pages that declined"); ax.set_ylim(0, 0.85)
plt.tight_layout(); plt.savefig("docs/img/chart2_decline_by_earlier_clicks.png", dpi=160); plt.close()

# Chart 3: model vs ranking by earlier clicks, pages with meaningful traffic
ks = [20, 50, 100]
mod = [p_at_k(m["score"].values, m["declined"].values, k) for k in ks]
rule = [p_at_k(m["clicks_prev_30d"].values, m["declined"].values, k) for k in ks]
fig, ax = plt.subplots(figsize=(7.2, 3.8))
x = np.arange(3)
b1 = ax.bar(x - w/2, mod, w, color=BLUE, label="Random Forest")
b2 = ax.bar(x + w/2, rule, w, color=GREY, label="Rank by earlier clicks")
label_bars(ax, b1); label_bars(ax, b2)
ax.axhline(m["declined"].mean(), color=INK, linestyle="--", linewidth=1, label="base rate")
ax.set_xticks(x); ax.set_xticklabels([f"precision@{k}" for k in ks])
ax.set_ylabel("precision (higher is better)"); ax.set_ylim(0, 1.05)
legend_on_top(ax, 3)
plt.tight_layout(); plt.savefig("docs/img/chart3_model_vs_rule.png", dpi=160); plt.close()

metrics = {"n_test_pages": int(len(te2)), "n_test_clients": int(te2["client_id"].nunique()),
           "base_decline_rate": float(te2["declined"].mean()), "base_rate_meaningful": float(base_m),
           "auc_model_meaningful": float(auc_m), "auc_earlier_clicks_meaningful": float(auc_r),
           "flagged_share": float(te2["high_risk"].mean()), "score_cutoff": float(cut2)}
with open("work/outputs/capstone_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Wrote the three charts to docs/img/ and the metrics to work/outputs/capstone_metrics.json")
print(sorted(os.listdir("docs/img")))

Wrote the three charts to docs/img/ and the metrics to work/outputs/capstone_metrics.json
['chart1_split_gap.png', 'chart2_decline_by_earlier_clicks.png', 'chart3_model_vs_rule.png']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.